# Exercise 5: Comparing rule-based and neural analysis

*Finite-State Morphology · version 0.1.0*

> Use both the TurkicNLP Apertium analyser and a neural morphological tagger (e.g., Stanza for Turkish via `Pipeline("tur", processors=["tokenize","pos","lemma"])`) to analyse the sentence *Gidebilseydik ne güzel olurdu*. Compare the analyses: where do they agree? Where do they differ? What does this tell you about the respective strengths and failure modes of each approach?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0"`. These notebooks compile lexc and twolc grammars, which needs the compiled `hfst` package; on PyPI it had wheels for Python 3.11 but not for 3.13 when checked on 19 October 2026, so use Python 3.11 or 3.12 if `pip` cannot find it. The Apertium transducers are downloaded once with `turkicnlp.download`.

*Gidebilseydik* 'if only we had been able to go' is *gid-ebil-se-y-di-k* (go-ABIL-COND-(y)-PST-1PL); *olurdu* 'it would be' is *ol-ur-du* (be-AOR-PST, 3SG). The Stanza cell loads the Stanza Turkish (IMST) models that TurkicNLP downloads, directly with `stanza.Pipeline`, so that the models are read from the local folder.

In [ ]:
import tempfile
from pathlib import Path
import hfst
import turkicnlp
from turkicnlp.resources.registry import ModelRegistry

def compile_grammar(lexc_text, twolc_text=None):
    """Compile a lexc lexicon (and optional twolc rules); return (analyser, generator) in lookup format."""
    work = Path(tempfile.mkdtemp())
    (work / "g.lexc").write_text(lexc_text, encoding="utf-8")
    gen = hfst.compile_lexc_file(str(work / "g.lexc"))
    if twolc_text:
        (work / "g.twolc").write_text(twolc_text, encoding="utf-8")
        hfst.compile_twolc_file(str(work / "g.twolc"), str(work / "g.twol.hfst"))
        stream, rules = hfst.HfstInputStream(str(work / "g.twol.hfst")), []
        while not stream.is_eof():
            rules.append(stream.read())
        stream.close()
        gen.compose_intersect(tuple(rules))
    gen.minimize()
    ana = hfst.HfstTransducer(gen)
    ana.invert()
    ana.minimize()
    for t in (ana, gen):
        t.convert(hfst.ImplementationType.HFST_OLW_TYPE)
    return ana, gen

def apertium(lang, script, kind="automorf"):
    """An Apertium transducer distributed with TurkicNLP (downloaded on first use)."""
    turkicnlp.download(lang, processors=["morph"])
    path = ModelRegistry.get_model_path(lang, "morph", "apertium", script=script) / f"{lang}.{kind}.hfst"
    stream = hfst.HfstInputStream(str(path))
    fst = stream.read()
    stream.close()
    return fst

lookup = lambda fst, s: sorted({out for out, _ in fst.lookup(s)})

In [ ]:
from turkicnlp import Pipeline

SENTENCE = "Gidebilseydik ne güzel olurdu."
ap = Pipeline("tur", processors=["tokenize", "morph"], tokenize_backend="rule", morph_backend="apertium")
for w in ap(SENTENCE).words:
    print(f"APERTIUM {w.text:<15} {w.lemma or '_':<8} {w.upos or '_':<6} {w.feats or '_'}")
fst = apertium("tur", "Latn")
for word in ["Gidebilseydik", "gidebilseydik", "olurdu"]:
    print(f"readings of {word}:", lookup(fst, word))

In [ ]:
try:
    turkicnlp.download("tur", processors=["tokenize", "pos", "lemma"])   # Stanza IMST models (once)
except Exception as exc:
    print("Download skipped (using local models):", type(exc).__name__)
try:
    import stanza
    nlp = stanza.Pipeline("tr", processors="tokenize,mwt,pos,lemma", model_dir=str(ModelRegistry.default_dir() / "stanza"),
                          download_method=None, verbose=False)
    for w in nlp(SENTENCE).iter_words():
        print(f"STANZA   {w.text:<15} {w.lemma or '_':<8} {w.upos or '_':<6} {w.feats or '_'}")
except Exception as exc:
    print("Stanza not available here:", type(exc).__name__, str(exc)[:120])

## Worked solution

Output of `experiments/ch06_exercises/run_ch06_exercises.py` on 19 October 2026 (TurkicNLP 0.5.0, HFST 3.16.0.1, Python 3.11):

```
APERTIUM Gidebilseydik   git      VERB   Evident=Fh|Mood=Cnd|Number=Plur|Person=1|Tense=Past
APERTIUM ne              ne       PRON   Case=Nom
APERTIUM güzel           güzel    NOUN   _
APERTIUM olurdu          ol       VERB   Aspect=Hab|Evident=Fh|Number=Sing|Person=3|Tense=Pres
APERTIUM .               .        PUNCT  _
readings of Gidebilseydik: []
readings of gidebilseydik: ['git<v><iv><abil><cond>+i<cop><ifi><p1><pl>', 'git<v><iv><pot><cond>+i<cop><ifi><p1><pl>']
readings of olurdu: ['ol<v><iv><aor>+i<cop><ifi><p3><sg>', 'ol<v><iv><ger_aor><nom>+i<cop><ifi><p3><sg>', 'ol<v><iv><gpr_aor><subst><nom>+i<cop><ifi><p3><sg>', 'olur<n><nom>+i<cop><ifi><p3><sg>']
```

```
  → Loading Stanza models for tur (tr) from /root/.turkicnlp/models/stanza/tr
Download skipped (using local models): ProxyError
STANZA   Gidebilse       git      VERB   Aspect=Perf|Mood=Des|Number=Sing|Person=3|Polarity=Pos|Tense=Pres
STANZA   ydik            i        AUX    Aspect=Perf|Mood=Ind|Number=Plur|Person=1|Tense=Past
STANZA   ne              ne       ADJ    _
STANZA   güzel           güzel    ADJ    _
STANZA   olurdu          ol       VERB   Aspect=Hab|Mood=Ind|Number=Sing|Person=3|Polarity=Pos|Tense=Past
STANZA   .               .        PUNCT  _
```

**Apertium.** The transducer has no reading for the capitalised *Gidebilseydik*; the pipeline analyses it anyway because it retries in lower case, where the transducer returns `git<v><iv><abil><cond>+i<cop><ifi><p1><pl>` (and the same with `<pot>`). The lemma *git* is right, including the *t*/*d* alternation, and so are conditional, past copula and 1PL. But the ability reading (`<abil>`) is lost in the mapping to UD features; there is no `Mood=Pot`. For *ne* (PRON) and *güzel* (NOUN) the heuristic selection picks readings that do not fit the exclamative 'how nice'. *olurdu* gets the right lemma and `Aspect=Hab`, but `Tense=Pres`, although *-du* is the past copula.

**Stanza (IMST models).** Stanza splits *Gidebilseydik* into two syntactic words, *Gidebilse* + *ydik* (AUX, lemma *i*), following the IMST treebank's treatment of the copula. It labels the conditional `Mood=Des`, does not mark ability either, and gives *Gidebilse* a default `Number=Sing|Person=3`; the person and number are on *ydik* (`Number=Plur|Person=1|Tense=Past`). It tags *güzel* as ADJ and gets `Tense=Past` on *olurdu*.

**Agreement and differences.** Both systems agree on the lemmas *git* and *ol*, on 1PL past for the first word and on the habitual for *olurdu*. They differ in tokenisation (one word or two syntactic words), in the label for the conditional, in the part of speech of *ne* and *güzel*, and in the tense of *olurdu*. The transducer's raw readings contain more grammatical information (ability) than either output shows, and every reading it returns is a form its grammar licenses; its errors come from the selection and the tag mapping. The neural tagger uses context, so it got the parts of speech and the tense right, but it can only produce what its treebank annotates, and it follows that treebank's conventions. One sentence shows failure modes; it does not rank the systems.

## Your turn — extension

1. Look up how the Turkish IMST treebank splits and annotates forms with *-ydi* and *-ymiş*, and compare with Stanza's output.
2. Try *Okuyamayacaktık* 'we would not have been able to read' with both systems.